# M05A · NCBI Entrez: GenBank & PubMed Retrieval
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M05_Databases_and_Scraping/M05A_Entrez_GenBank_PubMed_Solutions.ipynb)

⏱ ~120 min · Difficulty: ★★☆ · Prerequisites: M01A (Core Python), M02C (Biopython SeqIO & GenBank)


## 1. Learning Outcomes

By the end of this notebook, you will be able to:

- **Survey**: Navigate the primary biological databases (NCBI Nucleotide, Protein, Gene, PubMed, SRA, UniProt, PDB, KEGG) and interpret their accession numbering schemes.
- **Configure**: Set up NCBI Entrez credentials programmatically with mandatory email identification and secure API keys.
- **Query**: Execute programmatic searches across NCBI using E-utilities (`einfo`, `esearch`, `efetch`, `esummary`, `elink`) with advanced boolean field qualifiers.
- **Parse**: Retrieve and parse full GenBank records and MEDLINE biomedical abstracts into structured Python objects (`Bio.SeqIO`, `Bio.Medline`).
- **Cache**: Implement local disk caching to prevent redundant server traffic, accelerate data pipelines, and guarantee offline reproducibility.

## 2. Prerequisites


Before starting this notebook, you should be familiar with:
- Biopython `SeqIO` and `SeqRecord` data structures ([`M02C`](file:///modules/M02_File_Handling/M02C_Biopython_SeqIO_GenBank.ipynb)).
- Reading and writing files using Python's standard `pathlib.Path` ([`M02A`](file:///modules/M02_File_Handling/M02A_Text_Files.ipynb)).
- Fundamental Python dictionary traversal and list comprehensions ([`M01A`](file:///modules/M01_Python_Basics/M01A_Core_Python.ipynb)).


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn requests beautifulsoup4 lxml python-dotenv
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root robustly from modules/ or repo root
    p = Path.cwd()
    while p != p.parent and not (p / "COURSE_PLAN.md").exists():
        p = p.parent
    ROOT = p if (p / "COURSE_PLAN.md").exists() else Path.cwd().resolve()

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M05A"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import Entrez, SeqIO, Medline
import time

print(f"Python: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")
print(f"DATA_DIR: {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:  {OUT_DIR} (ready)")

## 5A.1 Biological Database Landscape & Accession Systems

### 4.1.1 Theory & Biological Context

Modern computational biology relies on centralized, curated biological repositories. Knowing where specific biological data resides and how it is identified is essential:

### The Primary Biological Database Ecosystem:

| Repository | Primary Host | What It Stores | Typical Accession / ID Formats | Example |
|---|---|---|---|---|
| **NCBI Nucleotide (GenBank / RefSeq)** | NCBI / NIH (USA) | DNA, mRNA, genomic assemblies | RefSeq: `NM_` (mRNA), `NR_` (ncRNA), `NC_` (chromosome), `NG_` (genomic region); GenBank: 1–2 letters + 5–6 digits | `NM_000546.6` (Human TP53 mRNA) |
| **NCBI Protein** | NCBI / NIH | Translated protein sequences | RefSeq: `NP_` (protein), `XP_` (predicted); GenBank: 3 letters + 5 digits | `NP_000537.3` (Human p53 protein) |
| **NCBI Gene** | NCBI / NIH | Gene-centric metadata, genomic coordinates, homologs | Integer Gene ID | `7157` (Human TP53) |
| **PubMed** | NCBI / NLM | Biomedical literature citations & abstracts | Integer PubMed ID (PMID) | `38123456` |
| **Sequence Read Archive (SRA)** | NCBI / EBI / DDBJ | High-throughput raw sequencing reads (FASTQ) | `SRR` / `ERR` / `DRR` + digits | `SRR1234567` |
| **UniProtKB (Swiss-Prot / TrEMBL)** | SIB / EBI / PIR | Curated protein sequence, 3D domains, post-translational modifications | 6 or 10 alphanumeric characters | `P04637` (P53_HUMAN) |
| **Protein Data Bank (PDB)** | wwPDB (RCSB, PDBe, PDBj) | 3D experimental macromolecular atomic coordinates | 4 alphanumeric characters (starting with digit) or extended PDB ID | `1TUP` (p53 core domain bound to DNA) |
| **KEGG** | Kanehisa Labs (Japan) | Molecular pathways, biological systems, drugs, diseases | Organism prefix + 5 digits | `hsa04115` (p53 signaling pathway - Homo sapiens) |

### INSDC Consortium:
GenBank (NCBI, USA), ENA (EMBL-EBI, Europe), and DDBJ (NIG, Japan) form the **International Nucleotide Sequence Database Collaboration (INSDC)**, exchanging sequence submissions daily.

### 4.1.2 Worked Example

In [ ]:
# 5A.1 Worked Example: Accession Syntax Validation
import re

def identify_biological_id(accession: str) -> str:
    """Categorizes biological accession format using regular expressions."""
    acc = accession.strip()
    if re.match(r"^NM_\d+(\.\d+)?$", acc):
        return "NCBI RefSeq mRNA"
    elif re.match(r"^NP_\d+(\.\d+)?$", acc):
        return "NCBI RefSeq Protein"
    elif re.match(r"^NC_\d+(\.\d+)?$", acc):
        return "NCBI RefSeq Chromosome"
    elif re.match(r"^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$", acc):
        return "UniProtKB Accession"
    elif re.match(r"^\d[A-Za-z0-9]{3}$", acc):
        return "PDB Structure ID"
    elif re.match(r"^[a-z]{3}\d{5}$", acc):
        return "KEGG Pathway ID"
    elif re.match(r"^\d{6,9}$", acc):
        return "PubMed ID (PMID) or NCBI Gene ID"
    return "Unknown Accession Format"

sample_accessions = ["NM_000546.6", "NP_000537.3", "P04637", "1TUP", "hsa04115", "38123456"]
for a in sample_accessions:
    print(f"{a:<15} -> {identify_biological_id(a)}")

### 4.1.3 ✏️ Try It: RefSeq Transcript Identifier

Write a function `is_refseq_transcript(acc: str) -> bool`:
- Return `True` if `acc` starts with `"NM_"` or `"NR_"` followed by digits and optional version `".N"`.
- Return `False` otherwise.

In [ ]:
def is_refseq_transcript(acc: str) -> bool:
    return bool(re.match(r"^N[MR]_\d+(\.\d+)?$", acc.strip()))

#### 💡 Explanation & Design Notes

RefSeq accessions follow strict prefix conventions (`NM_` for protein-coding mRNAs, `NR_` for non-coding RNAs, `NP_` for proteins). Distinguishing these prefixes prevents passing protein accessions to nucleotide endpoints.

In [ ]:
# ✅ Test — run me. No output means you passed.
assert is_refseq_transcript("NM_000546.6")
assert is_refseq_transcript("NR_046235.1")
assert not is_refseq_transcript("NP_000537.3")
assert not is_refseq_transcript("P04637")
print("Section 5A.1 passed ✔")

### 4.1.4 ⚠️ Common Mistakes

- **Confusing GenBank with RefSeq**: GenBank submissions are direct author deposits (redundant, variable curation). RefSeq records (starting with `N_`) are non-redundant, NCBI-curated reference benchmarks.
- **Stripping the accession version**: `NM_000546` always refers to the newest version, which can change over time. For scientific reproducibility, always specify the version: `NM_000546.6`.

## 5A.2 Entrez Configuration & Identification

### 4.2.1 Theory & Biological Context

NCBI Entrez (The Entrez Global Query Cross-Database Search System) provides programmatic access via **E-utilities**.

### NCBI Usage Policies:
1. **`Entrez.email` is MANDATORY**: You MUST supply a valid email address with every request. NCBI uses this to contact you if a runaway script creates abnormal server load before blocking your IP address.
2. **`Entrez.tool`**: Identifies your script or software pipeline.
3. **`Entrez.api_key`**:
   - Without an API key: Requests are limited to **3 requests per second**.
   - With an API key: Quota increases to **10 requests per second**.
   - API keys are obtained for free from your personal NCBI Account settings.

### 4.2.2 Worked Example

In [ ]:
# 5A.2 Worked Example: Configuring Entrez Credentials
# Always set email and tool
Entrez.email = os.environ.get("NCBI_EMAIL", "student.pcb@iiitd.ac.in")
Entrez.tool = "PCB_Course_Pipeline"

# Load API key from environment if available
api_key = os.environ.get("NCBI_API_KEY", None)
if api_key:
    Entrez.api_key = api_key
    print(f"Entrez configured with API key: {api_key[:4]}... (10 req/sec quota)")
else:
    print("Entrez configured without API key (Standard 3 req/sec quota)")

print("Entrez email set to:", Entrez.email)
print("Entrez tool set to: ", Entrez.tool)

### 4.2.3 ✏️ Try It: Entrez Credential Configurator

Write a function `configure_entrez(email: str, api_key: str = None) -> tuple[str, bool]`:
- Sets `Entrez.email` to `email`.
- If `api_key` is provided and non-empty, sets `Entrez.api_key` to `api_key` and returns `(email, True)`.
- If `api_key` is not provided, sets `Entrez.api_key = None` and returns `(email, False)`.

In [ ]:
def configure_entrez(email: str, api_key: str = None) -> tuple[str, bool]:
    Entrez.email = email
    if api_key:
        Entrez.api_key = api_key
        return (email, True)
    else:
        Entrez.api_key = None
        return (email, False)

#### 💡 Explanation & Design Notes

Encapsulating Entrez setup in a helper ensures credentials are valid and uniformly applied before making any API calls.

In [ ]:
# ✅ Test — run me. No output means you passed.
e, has_k = configure_entrez("tester@example.com", "secret123")
assert e == "tester@example.com" and has_k
assert Entrez.email == "tester@example.com"
assert Entrez.api_key == "secret123"

e, has_k = configure_entrez("tester@example.com")
assert not has_k and Entrez.api_key is None
print("Section 5A.2 passed ✔")

### 4.2.4 ⚠️ Common Mistakes

- **Hardcoding personal email or API key in shared Git repos**: Never push personal credentials to public GitHub repositories. Load them from environment variables via `os.environ.get()`.
- **Omitting Entrez.email**: NCBI will reject your requests with HTTP 400 Bad Request if `Entrez.email` is not provided.

## 5A.3 The E-Utilities Suite: einfo, esearch, efetch, esummary

### 4.3.1 Theory & Biological Context

NCBI provides 8 server-side CGI programs known collectively as **E-utilities** (Entrez Programming Utilities).

### The Core Five Utilities:

| Utility | Biopython Function | Primary Purpose | Input $\to$ Output |
|---|---|---|---|
| **EInfo** | `Entrez.einfo()` | List all accessible databases or retrieve statistics & field index for one database | None $\to$ DB list; or DB name $\to$ Field metadata |
| **ESearch** | `Entrez.esearch()` | Search database with a query string | Search term $\to$ List of primary IDs (UIDs) |
| **EFetch** | `Entrez.efetch()` | Download full records in desired format (FASTA, GenBank, MEDLINE, XML) | IDs $\to$ Full formatted text / XML records |
| **ESummary** | `Entrez.esummary()` | Retrieve brief document summaries (DocSums) without downloading whole sequence | IDs $\to$ Metadata dictionaries (Title, length, taxid) |
| **ELink** | `Entrez.elink()` | Discover cross-database links (e.g. from Gene ID to PubMed or PDB) | ID in DB A $\to$ Linked IDs in DB B |

### 4.3.2 Worked Example

In [ ]:
# 5A.3 Worked Example: Using einfo to inspect Entrez databases
try:
    # Query NCBI einfo to get list of available databases
    handle = Entrez.einfo()
    record = Entrez.read(handle)
    handle.close()
    
    db_list = record["DbList"]
    print(f"Total NCBI databases available: {len(db_list)}")
    print("Core databases:", [db for db in ["nucleotide", "protein", "gene", "pubmed", "structure"] if db in db_list])
    
    # Query database metadata for pubmed
    handle = Entrez.einfo(db="pubmed")
    db_info = Entrez.read(handle)
    handle.close()
    print(f"PubMed Description: {db_info['DbInfo']['Description']}")
    print(f"PubMed Total Records: {int(db_info['DbInfo']['Count']):,}")
except Exception as err:
    print(f"Network warning (using offline fallback simulation): {err}")

### 4.3.3 ✏️ Try It: EInfo Field Extractor

Write a function `parse_einfo_fields(db_info_dict: dict) -> list[str]`:
- Takes the dictionary returned by `Entrez.einfo(db=...)`.
- Extracts and returns a sorted list of all searchable field names (`Field['Name']`) from `db_info_dict['DbInfo']['FieldList']`.

In [ ]:
def parse_einfo_fields(db_info_dict: dict) -> list[str]:
    fields = [f["Name"] for f in db_info_dict.get("DbInfo", {}).get("FieldList", [])]
    return sorted(fields)

#### 💡 Explanation & Design Notes

Querying `einfo` dynamically reveals available field qualifiers for any NCBI database, enabling programmatic validation of user-submitted queries before sending them to `esearch`.

In [ ]:
# ✅ Test — run me. No output means you passed.
mock_info = {
    "DbInfo": {
        "FieldList": [
            {"Name": "AUTH", "Description": "Author Name"},
            {"Name": "TITL", "Description": "Title Word"},
            {"Name": "JOUR", "Description": "Journal Name"}
        ]
    }
}
fields = parse_einfo_fields(mock_info)
assert fields == ["AUTH", "JOUR", "TITL"]
print("Section 5A.3 passed ✔")

### 4.3.4 ⚠️ Common Mistakes

- **Forgetting to close the Entrez handle**: Every `Entrez` call returns an open HTTP stream handle. If not closed with `handle.close()`, sockets may remain open, triggering resource leaks.
- **Parsing with Entrez.read vs SeqIO.read**: `Entrez.read()` parses XML responses. For FASTA or GenBank text formats, parse with `Bio.SeqIO.read()`, not `Entrez.read()`.

## 5A.4 Fetching Sequences Programmatically with efetch

### 4.4.1 Theory & Biological Context

`Entrez.efetch` retrieves records directly from NCBI databases.

### Parameters of `Entrez.efetch`:
- `db`: Target database (`'nucleotide'`, `'protein'`, `'gene'`, `'pubmed'`).
- `id`: Accession number, UID, or comma-separated list of IDs.
- `rettype`: Return format:
  - For `nucleotide`: `'fasta'`, `'gb'` (GenBank flat file), `'gbwithparts'`.
  - For `protein`: `'fasta'`, `'gp'` (GenPept).
  - For `pubmed`: `'medline'`, `'abstract'`.
- `retmode`: Data encoding (`'text'` or `'xml'`).

### 4.4.2 Worked Example

In [ ]:
# 5A.4 Worked Example: Fetching and Parsing a GenBank Record
target_acc = "NM_000546.6"

try:
    print(f"Fetching {target_acc} from NCBI Nucleotide...")
    handle = Entrez.efetch(db="nucleotide", id=target_acc, rettype="gb", retmode="text")
    record = SeqIO.read(handle, "genbank")
    handle.close()
    
    print(f"Successfully retrieved online: {record.id}")
    print(f"Description: {record.description}")
    print(f"Sequence length: {len(record.seq):,} bp")
except Exception as e:
    print(f"Network call skipped ({e}). Falling back to local copy...")
    # Load from local repository copy in data/genbank/tp53_human.gb
    local_gb = DATA_DIR / "genbank" / "tp53_human.gb"
    record = SeqIO.read(local_gb, "genbank")
    print(f"Loaded from disk fallback: {record.id} ({len(record.seq):,} bp)")

### 4.4.3 ✏️ Try It: Record Summary Extractor

Write a function `extract_record_summary(record: SeqIO.SeqRecord) -> dict`:
- Extract and return a dictionary with:
  - `"id"`: `record.id`
  - `"length"`: `len(record.seq)`
  - `"molecule_type"`: `record.annotations.get("molecule_type", "unknown")`
  - `"num_features"`: `len(record.features)`

In [ ]:
def extract_record_summary(record: SeqIO.SeqRecord) -> dict:
    return {
        "id": record.id,
        "length": len(record.seq),
        "molecule_type": record.annotations.get("molecule_type", "unknown"),
        "num_features": len(record.features)
    }

#### 💡 Explanation & Design Notes

Biopython converts GenBank headers directly into `record.annotations` and features into `record.features`, enabling downstream biological analyses.

In [ ]:
# ✅ Test — run me. No output means you passed.
summary = extract_record_summary(record)
assert summary["length"] > 0
assert summary["num_features"] > 0
assert "id" in summary
print("Section 5A.4 passed ✔")

### 4.4.4 ⚠️ Common Mistakes

- **Using rettype='fasta' when feature annotations are needed**: FASTA format contains only description header and sequence. If you require gene boundaries, exons, or translations, use `rettype='gb'`.
- **Using SeqIO.read for multiple records**: `SeqIO.read()` assumes exactly ONE record in the handle. If multiple records are returned, use `SeqIO.parse()` instead.

## 5A.5 Advanced Searches: Field Tags & NCBI History Server

### 4.5.1 Theory & Biological Context

`Entrez.esearch` enables complex programmatic queries using NCBI field qualifiers and boolean logic:

### Search Field Qualifiers:
- `[Gene]` or `[GENE]`: Gene symbol (e.g. `TP53[Gene]`).
- `[Organism]` or `[ORGN]`: Scientific or common species name (e.g. `"Homo sapiens"[Organism]`).
- `[filter]`: Restrict to curated subsets (e.g. `refseq[filter]`, `biomol mrna[properties]`).
- `[Date - Publication]` or `[PDAT]`: Filter date ranges (e.g. `2020:2024[PDAT]`).

### NCBI History Server (`usehistory='y'`):
When searching queries that return thousands of records, do not download all IDs at once!
Set `usehistory="y"`: NCBI stores the results on their server, returning a `WebEnv` session token and `QueryKey`. You can then stream results in batches using `retstart` and `retmax`.

### 4.5.2 Worked Example

In [ ]:
# 5A.5 Worked Example: Advanced ESearch with Field Tags
search_term = "TP53[Gene] AND Homo sapiens[Organism] AND refseq[filter] AND biomol mrna[properties]"

try:
    print(f"Searching: {search_term}")
    handle = Entrez.esearch(db="nucleotide", term=search_term, retmax=5, usehistory="y")
    search_results = Entrez.read(handle)
    handle.close()
    
    total_count = int(search_results["Count"])
    id_list = search_results["IdList"]
    webenv = search_results["WebEnv"]
    query_key = search_results["QueryKey"]
    
    print(f"Total matching RefSeq mRNAs found: {total_count}")
    print(f"Top 5 UIDs: {id_list}")
    print(f"WebEnv token: {webenv[:20]}... QueryKey: {query_key}")
except Exception as e:
    print(f"Network search warning ({e}). Using mock result for offline continuity.")
    id_list = ["383209646", "2246031203", "1849009848"]
    total_count = 15

### 4.5.3 ✏️ Try It: RefSeq Query Formatter

Write a function `build_refseq_mrna_query(gene_symbol: str, organism: str) -> str`:
- Constructs a query string targeting RefSeq mRNAs for a gene in an organism.
- Format: `"{gene_symbol}[Gene] AND {organism}[Organism] AND refseq[filter] AND biomol mrna[properties]"`
- Return the formatted query string.

In [ ]:
def build_refseq_mrna_query(gene_symbol: str, organism: str) -> str:
    return f"{gene_symbol}[Gene] AND {organism}[Organism] AND refseq[filter] AND biomol mrna[properties]"

#### 💡 Explanation & Design Notes

Combining field qualifiers with boolean logic prevents noisy results such as non-coding fragments, patent sequences, and unannotated bacterial clones from contaminating your search.

In [ ]:
# ✅ Test — run me. No output means you passed.
q = build_refseq_mrna_query("BRCA1", "Homo sapiens")
assert 'BRCA1[Gene]' in q
assert '"Homo sapiens"[Organism]' in q or 'Homo sapiens[Organism]' in q
assert 'refseq[filter]' in q
print("Section 5A.5 passed ✔")

### 4.5.4 ⚠️ Common Mistakes

- **Searching broad terms without field qualifiers**: Searching `TP53` without `[Gene]` returns every paper, cloning vector, and synthetic construct that ever mentioned TP53 in its title or abstract.
- **Setting retmax > 10000 in a single request**: NCBI will truncate or reject single requests asking for massive record counts; always use `usehistory='y'` and paginate with `retstart`.

## 5A.6 PubMed Literature Searches & MEDLINE Parsing

### 4.6.1 Theory & Biological Context

PubMed contains over 36 million biomedical citations from MEDLINE, life science journals, and online books.

### Searching and Parsing PubMed:
1. Search with `Entrez.esearch(db="pubmed", term=...)` to retrieve PubMed IDs (PMIDs).
2. Fetch with `Entrez.efetch(db="pubmed", id=id_list, rettype="medline", retmode="text")`.
3. Parse with `Bio.Medline.parse(handle)`:
   - `record['PMID']`: Unique PubMed Identifier.
   - `record['TI']`: Article Title.
   - `record['AU']`: List of Author Names.
   - `record['JT']`: Full Journal Title.
   - `record['DP']`: Date of Publication.
   - `record['AB']`: Abstract text.

### 4.6.2 Worked Example

In [ ]:
# 5A.6 Worked Example: Querying PubMed and Parsing MEDLINE records
pubmed_query = "TP53 AND cuproptosis AND cancer"

try:
    print(f"Searching PubMed for: '{pubmed_query}'")
    handle = Entrez.esearch(db="pubmed", term=pubmed_query, retmax=3)
    p_results = Entrez.read(handle)
    handle.close()
    
    pmids = p_results["IdList"]
    print(f"Found PMIDs: {pmids}")
    
    if pmids:
        handle = Entrez.efetch(db="pubmed", id=pmids, rettype="medline", retmode="text")
        medline_records = list(Medline.parse(handle))
        handle.close()
        
        for idx, article in enumerate(medline_records, 1):
            print(f"[{idx}] PMID: {article.get('PMID')}")
            print(f"    Title:   {article.get('TI', 'N/A')[:90]}...")
            print(f"    Journal: {article.get('JT', 'N/A')}")
            print(f"    Date:    {article.get('DP', 'N/A')}")
except Exception as e:
    print(f"PubMed network access skipped ({e}). Demonstrating parser with local fixture...")
    # Mock medline parsing
    mock_medline = """PMID- 38123456
TI  - Mechanisms of p53 tumor suppression in response to copper-induced stress
AU  - Paul, P
AU  - Sharma, A
JT  - Journal of Biological Chemistry
DP  - 2024 Oct 15
AB  - Tumor suppressor p53 regulates metabolic homeostasis and cell death pathways.
"""
    import io
    medline_records = list(Medline.parse(io.StringIO(mock_medline)))
    print(f"Parsed {len(medline_records)} article: {medline_records[0]['TI']}")

### 4.6.3 ✏️ Try It: MEDLINE to DataFrame Converter

Write a function `medline_to_dataframe(articles: list[dict]) -> pd.DataFrame`:
- Convert a list of Medline records into a pandas DataFrame with columns:
  - `'pmid'`: string or integer PMID
  - `'title'`: article title
  - `'first_author'`: first author in `'AU'` list (or `"Unknown"` if none)
  - `'year'`: 4-digit publication year extracted from `'DP'` (e.g. `"2024"` from `"2024 Oct 15"`)
- Return the DataFrame.

In [ ]:
def medline_to_dataframe(articles: list[dict]) -> pd.DataFrame:
    rows = []
    for art in articles:
        pmid = art.get("PMID", "")
        title = art.get("TI", "")
        authors = art.get("AU", [])
        first_author = authors[0] if authors else "Unknown"
        dp = art.get("DP", "")
        # Extract 4-digit year
        match = re.search(r"(19\d\d|20\d\d)", dp)
        year = match.group(1) if match else "Unknown"
        rows.append({"pmid": pmid, "title": title, "first_author": first_author, "year": year})
    return pd.DataFrame(rows)

#### 💡 Explanation & Design Notes

`Bio.Medline` maps MEDLINE two-letter field tags directly into python dictionary keys. Standardizing these into a pandas DataFrame enables sorting, keyword filtering, and bibliographic export.

In [ ]:
# ✅ Test — run me. No output means you passed.
df_med = medline_to_dataframe(medline_records)
assert "pmid" in df_med.columns and "title" in df_med.columns
assert "first_author" in df_med.columns and "year" in df_med.columns
assert len(df_med) > 0
print("Section 5A.6 passed ✔")

### 4.6.4 ⚠️ Common Mistakes

- **Assuming all articles have abstracts ('AB')**: Review articles, editorials, and older citations frequently lack abstract text (`'AB'`), which will raise a `KeyError` if accessed via `art['AB']`. Always use `art.get('AB', '')`.
- **Hardcoding year string slice dp[:4]**: Some journals record dates as `Dec 2023` or `Summer 2022`. Using regex `re.search(r'\b(19\d\d|20\d\d)\b', dp)` reliably locates the 4-digit year.

## 5A.7 Data Persistence & Local Caching Strategies

### 4.7.1 Theory & Biological Context

The Golden Rule of Biological Data Retrieval:
> **Never query a remote public database twice for static data you can cache locally on disk.**

### Principles of Effective Caching:
1. **Cache Validation**: Before making any network call, check if the expected file already exists on disk.
2. **File Integrity**: Ensure the cached file is non-empty (`file.stat().st_size > 0`).
3. **Reproducibility**: When a remote record is fetched, save it immediately into `outputs/` with its explicit accession and version in the filename.

### 4.7.2 Worked Example

In [ ]:
# 5A.7 Worked Example: Production-Grade Caching Fetch Function
def fetch_and_cache_genbank(accession: str, cache_dir: Path) -> Path:
    """
    Checks if accession is cached locally; if not, fetches from NCBI and saves to disk.
    """
    cache_dir.mkdir(parents=True, exist_ok=True)
    target_path = cache_dir / f"{accession}.gb"
    
    if target_path.exists() and target_path.stat().st_size > 0:
        print(f"✔ Using local cached file: {target_path.name}")
        return target_path
    
    print(f"🌐 Fetching {accession} from NCBI Entrez...")
    try:
        handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text")
        content = handle.read()
        handle.close()
        
        target_path.write_text(content, encoding="utf-8")
        print(f"✔ Downloaded and cached to: {target_path.name} ({len(content):,} bytes)")
        return target_path
    except Exception as err:
        print(f"⚠️ Entrez fetch failed: {err}")
        # If local fallback exists in repo, copy it
        fallback = DATA_DIR / "genbank" / "tp53_human.gb"
        if fallback.exists():
            target_path.write_text(fallback.read_text(encoding="utf-8"), encoding="utf-8")
            print(f"✔ Copied repository fallback into cache: {target_path.name}")
            return target_path
        raise

cached_file = fetch_and_cache_genbank("NM_000546.6", OUT_DIR)

### 4.7.3 ✏️ Try It: Cache File Validator

Write a function `is_valid_cache_file(path: Path) -> bool`:
- Return `True` if `path` exists, is a file, and has file size greater than 100 bytes.
- Return `False` otherwise.

In [ ]:
def is_valid_cache_file(path: Path) -> bool:
    return path.exists() and path.is_file() and path.stat().st_size > 100

#### 💡 Explanation & Design Notes

Local caching insulates computational pipelines from intermittent network outages, adheres to NCBI fair usage policies, and accelerates automated testing suites.

In [ ]:
# ✅ Test — run me. No output means you passed.
assert is_valid_cache_file(cached_file)
assert not is_valid_cache_file(OUT_DIR / "non_existent.gb")
print("Section 5A.7 passed ✔")

### 4.7.4 ⚠️ Common Mistakes

- **Writing directly into data/ instead of outputs/**: The `data/` folder is read-only input. Always write retrieved files to `outputs/<MODULE_ID>/`.
- **Caching corrupt or empty files**: If a network disconnects halfway through a download, an empty 0-byte file might be written; always verify file size before treating as cached.

## 5A.8 GenBank Format Deep-Dive: Features, CDS & Exons

### 4.8.1 Theory & Biological Context

The GenBank flat file format is the gold-standard record structure of molecular biology.

### Anatomy of a GenBank Record:
1. **`LOCUS`**: Short locus name, sequence length, molecule type (`mRNA`, `DNA`), division (`PRI` = primate), date.
2. **`DEFINITION`**: Concise biological description of sequence and organism.
3. **`ACCESSION` & `VERSION`**: Unique ID and version number (e.g. `NM_000546.6`).
4. **`SOURCE` & `ORGANISM`**: Biological source and complete taxonomic lineage.
5. **`FEATURES`**: Structured table of genomic elements with coordinate spans:
   - `source`: Entire sequence span and tissue/isolate metadata.
   - `gene`: Gene symbol, synonyms, db_xref (`GeneID:7157`, `HGNC:11998`).
   - `CDS` (Coding Sequence): The exact translated region, reading frame, codon start, and `/translation` amino acid string.
   - `exon`: Exon boundary coordinates.
6. **`ORIGIN`**: The raw nucleotide sequence formatted in 6 columns of 10 bp blocks.

### 4.8.2 Worked Example

In [ ]:
# 5A.8 Worked Example: Inspecting GenBank Features Programmatically
gb_record = SeqIO.read(DATA_DIR / "genbank" / "tp53_human.gb", "genbank")

print(f"Locus Name:   {gb_record.name}")
print(f"Description:  {gb_record.description}")
print(f"Taxonomy:     {' > '.join(gb_record.annotations.get('taxonomy', [])[:5])}")

# Search for CDS features
cds_features = [f for f in gb_record.features if f.type == "CDS"]
print(f"Found {len(cds_features)} CDS feature(s):")
for cds in cds_features:
    print(f"  Location:    {cds.location}")
    print(f"  Gene symbol: {cds.qualifiers.get('gene', ['N/A'])[0]}")
    print(f"  Protein ID:  {cds.qualifiers.get('protein_id', ['N/A'])[0]}")
    translation = cds.qualifiers.get('translation', [''])[0]
    print(f"  Translation: {translation[:30]}... ({len(translation)} aa)")

### 4.8.3 ✏️ Try It: Exon Span Extractor

Write a function `get_all_exon_spans(record: SeqIO.SeqRecord) -> list[tuple[int, int]]`:
- Iterate over all features with `type == "exon"`.
- For each exon, extract `(int(f.location.start), int(f.location.end))`.
- Return a sorted list of these coordinate tuples.

In [ ]:
def get_all_exon_spans(record: SeqIO.SeqRecord) -> list[tuple[int, int]]:
    exons = []
    for f in record.features:
        if f.type == "exon":
            exons.append((int(f.location.start), int(f.location.end)))
    return sorted(exons)

#### 💡 Explanation & Design Notes

Biopython converts GenBank 1-based inclusive coordinates automatically into Python 0-based half-open intervals `[start, end)`. This means slicing `record.seq[start:end]` extracts the exact exon sequence directly.

In [ ]:
# ✅ Test — run me. No output means you passed.
exons = get_all_exon_spans(gb_record)
assert len(exons) > 0
assert all(start < end for start, end in exons)
print("Section 5A.8 passed ✔")

### 4.8.4 ⚠️ Common Mistakes

- **Assuming 1-based coordinates in Biopython**: GenBank flat files display 1-based coordinates (e.g. `1..2500`), but Biopython's `f.location.start` is 0-based. Slicing with `record.seq[f.location.start:f.location.end]` works out of the box.
- **Accessing qualifiers as strings instead of lists**: `feature.qualifiers` maps keys to LISTS of strings (e.g. `{'gene': ['TP53']}`). Always access via `cds.qualifiers['gene'][0]`.

### Practice Exercise 5A.1 (★) — Extracting All CDS Translations from Bacterial Lac Operon

**Problem**: Write a function `extract_cds_translations(gb_path: Path) -> dict[str, str]`:
- Parse the GenBank file at `gb_path` (e.g. `data/genbank/ecoli_lac_operon.gb`).
- For every feature of type `"CDS"`:
  - Extract the gene name from `qualifiers.get("gene", [None])[0]`. If gene name is not present, use the `protein_id` or locus tag.
  - Extract the protein translation string from `qualifiers["translation"][0]`.
- Return a dictionary mapping `{gene_or_id: translation_string}`.


- **Input**: Path to a GenBank file
- **Expected Output**: Dictionary of CDS identifiers mapped to their translated amino acid sequences

In [ ]:
def extract_cds_translations(gb_path: Path) -> dict[str, str]:
    record = SeqIO.read(gb_path, "genbank")
    cds_dict = {}
    for f in record.features:
        if f.type == "CDS":
            gene = f.qualifiers.get("gene", [None])[0]
            if not gene:
                gene = f.qualifiers.get("protein_id", f.qualifiers.get("locus_tag", ["unknown"]))[0]
            translation = f.qualifiers.get("translation", [""])[0]
            if gene and translation:
                cds_dict[gene] = translation
    return cds_dict

#### 💡 Solution Walkthrough (5A.1)

This exercise demonstrates extracting all protein products encoded by a multi-cistronic bacterial operon directly from its GenBank record.

In [ ]:
# ✅ Test — run me. No output means you passed.
lac_gb = DATA_DIR / "genbank" / "ecoli_lac_operon.gb"
translations = extract_cds_translations(lac_gb)
assert len(translations) >= 3, "Should contain lacZ, lacY, lacA proteins"
assert any("lacZ" in k or "lac" in k.lower() for k in translations)
print("Ex 5A.1 passed ✔")

### Practice Exercise 5A.2 (★★) — Automated PubMed Literature Mining to CSV

**Problem**: Write a function `mine_pubmed_literature(query: str, max_records: int, out_csv: Path) -> Path`:
- Query PubMed for `query` using `Entrez.esearch`.
- If results are returned, fetch MEDLINE records using `Entrez.efetch(rettype="medline")`.
- Parse articles and extract `[pmid, title, first_author, journal, year]`.
- If network fails or yields no results, construct a mock table of 3 relevant literature records.
- Save results as CSV to `out_csv`.
- Return `out_csv`.


- **Input**: Query string, max_records integer, and output CSV Path
- **Expected Output**: Path to saved CSV containing literature metadata

In [ ]:
def mine_pubmed_literature(query: str, max_records: int, out_csv: Path) -> Path:
    rows = []
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_records)
        search_res = Entrez.read(handle)
        handle.close()
        pmids = search_res.get("IdList", [])
        if pmids:
            handle = Entrez.efetch(db="pubmed", id=pmids, rettype="medline", retmode="text")
            articles = list(Medline.parse(handle))
            handle.close()
            for art in articles:
                pmid = art.get("PMID", "")
                title = art.get("TI", "")
                authors = art.get("AU", [])
                first_author = authors[0] if authors else "Unknown"
                journal = art.get("JT", "")
                dp = art.get("DP", "")
                m = re.search(r"(19\d\d|20\d\d)", dp)
                year = m.group(1) if m else "Unknown"
                rows.append({"pmid": pmid, "title": title, "first_author": first_author, "journal": journal, "year": year})
    except Exception as err:
        print(f"Notice: PubMed search fallback ({err})")

    if not rows:
        # Fallback offline records
        rows = [
            {"pmid": "38123456", "title": "Targeting p53 pathways in oncology", "first_author": "Paul, P", "journal": "Nature Reviews Cancer", "year": "2024"},
            {"pmid": "37987654", "title": "Mechanisms of drug resistance in cancer", "first_author": "Sharma, A", "journal": "Cancer Cell", "year": "2023"},
            {"pmid": "36543210", "title": "Genomic profiling of tumor suppressors", "first_author": "Smith, J", "journal": "Cell", "year": "2022"}
        ]
        
    df = pd.DataFrame(rows)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    return out_csv

#### 💡 Solution Walkthrough (5A.2)

This function demonstrates a resilient bibliographic harvesting pipeline with graceful offline simulation, essential for automated testing and CI runners.

In [ ]:
# ✅ Test — run me. No output means you passed.
out_lit = OUT_DIR / "pubmed_results.csv"
res_p = mine_pubmed_literature("TP53 apoptosis", 3, out_lit)
assert res_p.exists() and res_p.stat().st_size > 50
df_res = pd.read_csv(res_p)
assert "pmid" in df_res.columns and len(df_res) >= 3
print("Ex 5A.2 passed ✔")

### Practice Exercise 5A.3 (★★★) — End-to-End Caching Sequence Extraction Pipeline

**Problem**: Write a function `retrieve_and_translate_gene(accession: str, out_dir: Path) -> dict`:
- Check if `{accession}.gb` exists in `out_dir`. If not, fetch from NCBI Nucleotide and cache it.
- If network fails, copy fallback from `data/genbank/tp53_human.gb` or `data/genbank/insulin_protein.gp`.
- Parse the cached GenBank file with `SeqIO.read`.
- Locate the main `"CDS"` feature.
- Extract:
  - `"accession"`: accession string
  - `"gene"`: gene symbol
  - `"product"`: product name
  - `"cds_length_nt"`: length of coding sequence in nucleotides
  - `"protein_translation"`: translated amino acid sequence
- Return the dictionary.


- **Input**: Accession string and output cache directory Path
- **Expected Output**: Dictionary containing extracted gene annotations and translation

In [ ]:
def retrieve_and_translate_gene(accession: str, out_dir: Path) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    target_file = out_dir / f"{accession}.gb"
    
    if not (target_file.exists() and target_file.stat().st_size > 0):
        try:
            handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text")
            content = handle.read()
            handle.close()
            target_file.write_text(content, encoding="utf-8")
        except Exception:
            # Fallback
            fb = DATA_DIR / "genbank" / "tp53_human.gb"
            target_file.write_text(fb.read_text(encoding="utf-8"), encoding="utf-8")
            
    record = SeqIO.read(target_file, "genbank")
    cds_feats = [f for f in record.features if f.type == "CDS"]
    if not cds_feats:
        raise ValueError("No CDS feature found in record")
    
    main_cds = cds_feats[0]
    gene = main_cds.qualifiers.get("gene", ["Unknown"])[0]
    prod = main_cds.qualifiers.get("product", ["Unknown"])[0]
    trans = main_cds.qualifiers.get("translation", [""])[0]
    cds_len = len(main_cds.extract(record.seq))
    
    return {
        "accession": accession,
        "gene": gene,
        "product": prod,
        "cds_length_nt": cds_len,
        "protein_translation": trans
    }

#### 💡 Solution Walkthrough (5A.3)

This pipeline executes the complete workflow: caching layer -> sequence fetch -> GenBank parser -> CDS feature location -> nucleotide length calculation -> protein translation extraction.

In [ ]:
# ✅ Test — run me. No output means you passed.
pipeline_res = retrieve_and_translate_gene("NM_000546.6", OUT_DIR)
assert pipeline_res["gene"] == "TP53"
assert len(pipeline_res["protein_translation"]) == 393
assert pipeline_res["cds_length_nt"] == 1182  # 393 * 3 + 3 (stop) = 1182
print("Ex 5A.3 passed ✔")

## 6. Summary & Cheat Sheet

NCBI Entrez provides programmatic access to the world's primary biological repositories. By configuring Entrez credentials, constructing targeted search queries with field qualifiers, caching records locally, and parsing with Biopython, you can automate genomic discovery.

### Quick Reference Table

| Concept / Operation | Syntax / Function | Biological Application | Key Parameter / Note |
| --- | --- | --- | --- |
| Entrez Email | Entrez.email = 'user@example.com' | Mandatory identification | Required by NCBI |
| Entrez API Key | Entrez.api_key = os.environ.get(...) | Raises rate limit to 10 req/s | Never commit to git |
| Database Info | handle = Entrez.einfo(db=...) | Inspect searchable fields | Parse with Entrez.read() |
| Search | handle = Entrez.esearch(db=..., term=...) | Search IDs by query | Use field tags like [Gene] |
| Fetch Sequence | handle = Entrez.efetch(db=..., id=..., rettype='gb') | Download full record | Parse with SeqIO.read() |
| Fetch Literature | handle = Entrez.efetch(db='pubmed', rettype='medline') | Download citations | Parse with Medline.parse() |
| Caching | if path.exists() and path.stat().st_size > 0: | Prevent redundant downloads | Save to outputs/<MXX>/ |

## 7. Further Reading & Resources

- [Biopython Tutorial: Chapter 9 (Accessing NCBI Entrez)](https://biopython.org/DIST/docs/tutorial/Tutorial.html#sec127): Official Biopython E-utilities guide.
- [NCBI Entrez Programming Utilities (E-utilities) Help](https://www.ncbi.nlm.nih.gov/books/NBK25501/): Comprehensive reference manual by the National Center for Biotechnology Information.
- [RefSeq: A Curated Non-Redundant Database](https://academic.oup.com/nar/article/44/D1/D733/2502646): O'Leary et al., Nucleic Acids Research.